# Large Scale Deep Learning, Constructor University
## Home assignment #4: Self-supervised Learning in Audio
### Grading and Penalties

The maximum possible score for the assignment is **10 points**. Submissions after the stated deadline are not allowed.

You must complete the assignment independently. ?Similar? solutions are considered plagiarism, and all involved students (including those whose work was copied) cannot receive more than 0 points for it. If you found a solution to any part of the assignment in an open source, you must provide a link to that source in a separate block at the end of your work (most likely you won?t be the only one who found it, so to avoid suspicion of plagiarism, a source link is required).

Inefficient code implementation may negatively affect the grade. The grade may also be reduced for poorly readable code and poorly formatted plots. All answers must be accompanied by code or comments explaining how they were obtained.

### About the Assignment

In this homework, we will train audio models for different representations of audio data. As our dataset, we will use AudioMNIST: it contains 60 speakers pronouncing the digits 0?9. Our main task is digit classification, but in addition, these data will allow us to explore voice biometrics.

This assignment does not require a separate report, but we expect you to clearly and properly describe all of your experiments in the notebook?s text cells. If the grader sees bare numbers without your explanation and analysis, they reserve the right to penalize you at their discretion.

You can download the data from GitHub:

In [ ]:
!git clone https://github.com/soerenab/AudioMNIST.git

In [ ]:
import os
import random
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from IPython.display import Audio

plt.rcParams.update({'font.size': 14})

Below you can see an example of how to load audio files from disk if you are working with audio for the first time.

In [ ]:
root = 'AudioMNIST/data'
rand_speaker = random.choice(os.listdir(root))
rand_audio = random.choice(os.listdir(os.path.join(root, rand_speaker)))
print(f'Speaker #{rand_speaker}, class #{rand_audio[0]}')

waveform, sample_rate = torchaudio.load(
    os.path.join(root, rand_speaker, rand_audio), normalize=True
)
print(f'Sample rate: {sample_rate}, waveform shape: {waveform.shape}')
Audio(waveform, rate=sample_rate)

The sampling rate in our data is quite high (48 kHz), while 16 kHz is usually sufficient in audio models to process our audio tracks without losing quality. We can resample the recordings as follows:

In [ ]:
new_sample_rate = 16000
resample = T.Resample(orig_freq=sample_rate, new_freq=new_sample_rate)
resampled_waveform = resample(waveform)
Audio(resampled_waveform, rate=new_sample_rate)

One of the popular ways to represent audio data is to convert it to a log mel-spectrogram. The output is an image where one dimension corresponds to time and the other to frequencies. To speed up processing, you can move the module to the GPU (everything will work, since the transforms inherit from `nn.Module`). Also note that the values of the logarithm of the spectrogram are shifted relative to zero, so it makes sense to normalize the spectrogram before feeding it into the neural network.

In [ ]:
class LogMelSpectrogram(T.MelSpectrogram):
    def __init__(self, eps=1e-8, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def forward(self, waveform):
        return (super().forward(waveform) + self.eps).log()

In [ ]:
spectrogram = LogMelSpectrogram(sample_rate=16000, n_mels=64)
logmel_spec = spectrogram(resampled_waveform.squeeze())
print(f'Spectrogram shape: {logmel_spec.shape}')
print(f'Values range: ({logmel_spec.min(), logmel_spec.max()})')

plt.imshow(logmel_spec.numpy())
plt.ylabel('Mels')
plt.xlabel('Timesteps')
plt.show()

## Task 0. Preparation

**0 points**

* Let?s discuss how to split the data into train and test. Since we want to experiment with voice biometrics, it makes sense to split the data by speakers. Set aside one third of the speakers as the test set. The remaining speakers will serve as train and validation. Split them in an 80:20 ratio. This time you don?t need to split the data by speakers; instead, perform a stratified split (both by speakers and by classes within speakers).

* In this assignment, we will work with two representations of audio data: raw audio waveforms and log mel-spectrograms. We will process them using 1D and 2D convolutions, respectively. Implement two small neural networks: one with a 1D encoder and one with a 2D encoder. It will be sufficient to stack several convolutional layers, since our data is quite simple and does not require SOTA architectures. We recommend using 1D convolutions with dilation, stride, and a relatively large kernel size.

## Task 1. Supervised Models

**1 point**

Independently train the 1D encoder on raw waveforms and the 2D encoder on log mel-spectrograms for digit classification. Use accuracy as the evaluation metric. Assess the performance on the test set.

## Task 2. Multi-format contrastive learning

**4 points**

Now let's move to unsupervised pretraining. You need to implement a method that contrasts embeddings from two formats of audio data: [paper](https://arxiv.org/abs/2103.06508). Use the same encoder architectures as in the supervised training. We recommend checking the discussion of the second homework to correctly implement the contrastive loss.

Choose one type of augmentation for the raw audio waveforms and one for the spectrograms, and train 4 different models (no augmentations, augmentation only for waveforms, augmentation only for spectrograms, and augmentation for both). Measure linear probe performance in three ways (only on the 1D encoder outputs, only on the 2D encoder outputs, and on the concatenation of outputs) and select the best model by validation. That model will be used in subsequent experiments. Evaluate performance on the test set.

To save time, we will not fine-tune the models and will limit ourselves to linear probing only.

## Task 3. Visualization

**2 points**

We now have 4 models: two 1D encoders and two 2D encoders trained in supervised and self-supervised manner. For each of the 4 models, plot a t-SNE of the embeddings for the test set (the test split you set aside at the beginning of the assignment). For each model produce two plots: one where embeddings are colored by speaker (each speaker a different color) and another where embeddings are colored by digit.

## Task 4. Voice Biometrics

**3 points**

Since a model for voice biometrics must be able to handle new speakers, non-parametric classifiers such as kNN are very often used for this task.

We will evaluate voice biometrics as follows: take the test speakers and generate embeddings of their recordings using the model. Then split these embeddings into two equal-sized subsets: we will train kNN on one subset and test it on the other. As with the train/validation split, you must perform a stratified split both by speakers and by digits.

Select the optimal number of neighbors **k** via cross-validation on the training subset (i.e., the subset taken from the test set, not the original training data), and compute the speaker classification accuracy for all 4 models.

## Task 5. Bonus

**1 point**

Fine-tune the self-supervised models on digit classification. Compare the classification and voice-biometrics performance with the supervised models and with the linear-probe results